In [5]:
import numpy as np
import pandas as pd


def _date_window(start_date, end_date, lookback_days=0):
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)
    start_day = start.normalize()
    end_day = end.normalize()
    end_query = end if end != end.normalize() else end + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
    query_start = start_day - pd.Timedelta(days=lookback_days)
    return (
        start_day.strftime("%Y-%m-%d"),
        end_day.strftime("%Y-%m-%d"),
        query_start.strftime("%Y-%m-%d %H:%M:%S"),
        end_query.strftime("%Y-%m-%d %H:%M:%S"),
    )


def _clean_result(df):
    columns = ["date", "instrument", "factor"]
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=columns)
    df = df[columns].copy()
    df["date"] = pd.to_datetime(df["date"])
    df["factor"] = pd.to_numeric(df["factor"], errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=columns)
    df = df.groupby(["date", "instrument"], as_index=False)["factor"].mean()
    return df.sort_values(["date", "instrument"]).reset_index(drop=True)


def main(data_source=None, start_date=None, end_date=None):
    from bigquant import dai

    start_day, end_day, query_start, end_query = _date_window(start_date, end_date, lookback_days=5)
    end_next = (pd.to_datetime(end_day) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    sql = f"""
    WITH base AS (
        SELECT
            CAST(f.date AS DATE) AS date,
            f.instrument,
            COALESCE(CAST(e.industry_level1_code AS VARCHAR), 'UNKNOWN') AS industry_level1_code,
            f.roa_avg_ttm,
            f.roe_avg_ttm,
            f.gross_profit_rate_ttm,
            f.net_profit_rate_ttm,
            CASE WHEN f.pe_ttm > 0 THEN 1.0 / f.pe_ttm ELSE NULL END AS earn_yield,
            CASE WHEN f.pb > 0 THEN 1.0 / f.pb ELSE NULL END AS book_to_price,
            CASE WHEN f.ps_ttm > 0 THEN 1.0 / f.ps_ttm ELSE NULL END AS sales_to_price,
            -1.0 * f.debt_to_asset_lf AS low_leverage,
            -1.0 * e.SIZE AS low_size,
            f.netflow_amount_rate_main AS main_flow
        FROM bigalpha_2026_factorlib f
        LEFT JOIN bigalpha_2026_exposure e
          ON CAST(f.date AS DATE) = CAST(e.date AS DATE) AND f.instrument = e.instrument
        WHERE f.date >= '{query_start}' AND f.date <= '{end_query}'
          AND COALESCE(f.list_days, 0) >= 120
    ),
    z AS (
        SELECT
            date,
            instrument,
            industry_level1_code,
            COALESCE((roa_avg_ttm - AVG(roa_avg_ttm) OVER (PARTITION BY date)) / NULLIF(STDDEV_SAMP(roa_avg_ttm) OVER (PARTITION BY date), 0.0), 0.0) AS z_roa,
            COALESCE((roe_avg_ttm - AVG(roe_avg_ttm) OVER (PARTITION BY date)) / NULLIF(STDDEV_SAMP(roe_avg_ttm) OVER (PARTITION BY date), 0.0), 0.0) AS z_roe,
            COALESCE((gross_profit_rate_ttm - AVG(gross_profit_rate_ttm) OVER (PARTITION BY date)) / NULLIF(STDDEV_SAMP(gross_profit_rate_ttm) OVER (PARTITION BY date), 0.0), 0.0) AS z_gross_margin,
            COALESCE((net_profit_rate_ttm - AVG(net_profit_rate_ttm) OVER (PARTITION BY date)) / NULLIF(STDDEV_SAMP(net_profit_rate_ttm) OVER (PARTITION BY date), 0.0), 0.0) AS z_net_margin,
            COALESCE((earn_yield - AVG(earn_yield) OVER (PARTITION BY date)) / NULLIF(STDDEV_SAMP(earn_yield) OVER (PARTITION BY date), 0.0), 0.0) AS z_earn_yield,
            COALESCE((book_to_price - AVG(book_to_price) OVER (PARTITION BY date)) / NULLIF(STDDEV_SAMP(book_to_price) OVER (PARTITION BY date), 0.0), 0.0) AS z_book_to_price,
            COALESCE((sales_to_price - AVG(sales_to_price) OVER (PARTITION BY date)) / NULLIF(STDDEV_SAMP(sales_to_price) OVER (PARTITION BY date), 0.0), 0.0) AS z_sales_to_price,
            COALESCE((low_leverage - AVG(low_leverage) OVER (PARTITION BY date)) / NULLIF(STDDEV_SAMP(low_leverage) OVER (PARTITION BY date), 0.0), 0.0) AS z_low_leverage,
            COALESCE((low_size - AVG(low_size) OVER (PARTITION BY date)) / NULLIF(STDDEV_SAMP(low_size) OVER (PARTITION BY date), 0.0), 0.0) AS z_low_size,
            COALESCE((main_flow - AVG(main_flow) OVER (PARTITION BY date)) / NULLIF(STDDEV_SAMP(main_flow) OVER (PARTITION BY date), 0.0), 0.0) AS z_main_flow
        FROM base
    ),
    raw AS (
        SELECT
            date,
            instrument,
            industry_level1_code,
            0.70 * z_roa
            + 0.70 * z_roe
            + 0.55 * z_gross_margin
            + 0.35 * z_net_margin
            + 0.80 * z_earn_yield
            + 0.55 * z_book_to_price
            + 0.35 * z_sales_to_price
            + 0.45 * z_low_leverage
            + 0.25 * z_low_size
            + 0.20 * z_main_flow AS raw_factor
        FROM z
    ),
    industry_neutral AS (
        SELECT
            date,
            instrument,
            raw_factor - AVG(raw_factor) OVER (PARTITION BY date, industry_level1_code) AS raw_factor
        FROM raw
    ),
    standardized AS (
        SELECT
            date,
            instrument,
            raw_factor,
            AVG(raw_factor) OVER (PARTITION BY date) AS mu,
            STDDEV_SAMP(raw_factor) OVER (PARTITION BY date) AS sigma
        FROM industry_neutral
        WHERE raw_factor IS NOT NULL
    )
    SELECT
        date,
        instrument,
        (raw_factor - mu) / NULLIF(sigma, 0.0) AS factor
    FROM standardized
    WHERE date >= '{start_day}' AND date <= '{end_day}'
      AND sigma > 0
    ORDER BY date, instrument
    """

    df = _clean_result(dai.query(sql, filters={"date": [query_start[:10], end_next]}).df())
    if len(df) > 0:
        return df

    fallback_sql = f"""
    WITH base AS (
        SELECT
            CAST(date AS DATE) AS date,
            instrument,
            0.35 * COALESCE(momentum_5, 0.0)
            + 0.35 * COALESCE(reversal_5, 0.0)
            + 0.25 * COALESCE(roe_avg_ttm, 0.0)
            + 0.20 * COALESCE(roa_avg_ttm, 0.0)
            + 0.15 * CASE WHEN pe_ttm > 0 THEN 1.0 / pe_ttm ELSE 0.0 END
            - 0.20 * COALESCE(volatility_5, 0.0)
            + 0.000001 * COALESCE(close, 0.0) AS raw_factor
        FROM bigalpha_2026_factorlib
        WHERE date >= '{start_day}' AND date < '{end_next}'
    ),
    standardized AS (
        SELECT
            date,
            instrument,
            raw_factor,
            AVG(raw_factor) OVER (PARTITION BY date) AS mu,
            STDDEV_SAMP(raw_factor) OVER (PARTITION BY date) AS sigma
        FROM base
    )
    SELECT
        date,
        instrument,
        CASE
            WHEN sigma > 0 THEN (raw_factor - mu) / sigma
            ELSE 0.0
        END AS factor
    FROM standardized
    WHERE date >= '{start_day}' AND date <= '{end_day}'
    ORDER BY date, instrument
    """

    return _clean_result(dai.query(fallback_sql, filters={"date": [start_day, end_next]}).df())


(Empty DataFrame
 Columns: [date, instrument, factor]
 Index: [],
 (0, 3),
 Index(['date', 'instrument', 'factor'], dtype='object'))